<a href="https://colab.research.google.com/github/Danny3636/Generative-AI-Tasks/blob/main/Hedge_Fund_Holdings_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 Institutional Flow Tracker — RAG Pipeline
## SEC 13F Hedge Fund Holdings Intelligence System

**Domain**: SEC 13F institutional equity holdings disclosures
**Data Source**: SEC Bulk 13F Datasets (pre-processed TSV)
**Advanced RAG**: Hybrid retrieval (BM25 + Vector) + Cross-encoder reranking + Hierarchical chunking
**LLM**: Llama 3.3 70B via Groq (free tier)

---

## 1. Setup

In [1]:
!pip install -q groq chromadb sentence-transformers rank-bm25
!pip install -q requests beautifulsoup4 lxml tqdm
!pip install -q scikit-learn matplotlib seaborn plotly ipywidgets
!pip install -q yfinance

print("All dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [2]:
import os, json, time, re, hashlib, zipfile, io, warnings
from datetime import datetime, timedelta
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict
from io import BytesIO

import requests
import pandas as pd
import numpy as np
import yfinance as yf
from tqdm import tqdm

import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from groq import Groq

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, HTML, Markdown
warnings.filterwarnings("ignore")
print("All imports loaded")

All imports loaded


## 2. Configuration

In [3]:
@dataclass
class Config:
    SEC_USER_AGENT: str = "InstitutionalFlowTracker your_email@warwick.ac.uk"
    SEC_BASE_URL: str = "https://data.sec.gov"

    # Bulk dataset URLs — SEC publishes these quarterly
    SEC_BULK_URLS: Dict[str, str] = field(default_factory=lambda: {
        "2025-Q4": "https://www.sec.gov/files/structureddata/data/form-13f-data-sets/01dec2025-28feb2026_form13f.zip",
        "2025-Q3": "https://www.sec.gov/files/structureddata/data/form-13f-data-sets/01sep2025-30nov2025_form13f.zip",
        "2025-Q2": "https://www.sec.gov/files/structureddata/data/form-13f-data-sets/01jun2025-31aug2025_form13f.zip",
        "2025-Q1": "https://www.sec.gov/files/structureddata/data/form-13f-data-sets/01mar2025-31may2025_form13f.zip",
    })

    QUARTERS_TO_FETCH: List[str] = field(default_factory=lambda: ["2025-Q4", "2025-Q3"])

    TARGET_FUNDS: Dict[int, str] = field(default_factory=lambda: {
        1067983: "Berkshire Hathaway (Buffett)",
        1336528: "Bridgewater Associates",
        1423053: "Citadel Advisors",
        1037389: "Renaissance Technologies",
        1336326: "Pershing Square Capital",
        1061768: "Third Point LLC",
        1273087: "Viking Global Investors",
        1167483: "Two Sigma Investments",
        1364742: "D.E. Shaw & Co",
        1510387: "Tiger Global Management",
        1535392: "Coatue Management",
        1173334: "Lone Pine Capital",
        1040273: "Appaloosa Management",
        1079114: "Greenlight Capital",
        921669:  "Icahn Capital",
        1029160: "Soros Fund Management",
        1582507: "Point72 Asset Management",
        1345471: "Baupost Group",
        1536071: "Elliott Investment Management",
        1649339: "Millennium Management",
        1510761: "Dragoneer Investment Group",
        1040971: "Maverick Capital",
        1111830: "Jana Partners",
        1279708: "Paulson & Co",
        1159159: "ValueAct Capital",
    })

    EMBEDDING_MODEL: str = "all-MiniLM-L6-v2"
    RERANKER_MODEL: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    TOP_K_RETRIEVAL: int = 20
    TOP_K_RERANK: int = 8
    TOP_K_BASELINE: int = 8
    BM25_WEIGHT: float = 0.4
    VECTOR_WEIGHT: float = 0.6
    GROQ_API_KEY: str = ""
    LLM_MODEL: str = "llama-3.3-70b-versatile"
    LLM_TEMPERATURE: float = 0.1
    LLM_MAX_TOKENS: int = 1000

config = Config()
print(f"Config: {len(config.TARGET_FUNDS)} funds, quarters: {config.QUARTERS_TO_FETCH}")

Config: 25 funds, quarters: ['2025-Q4', '2025-Q3']


In [4]:
from google.colab import userdata
config.GROQ_API_KEY = userdata.get("GROQ_API_KEY")
config.SEC_USER_AGENT = "InstitutionalFlowTracker your_email@warwick.ac.uk"  # UPDATE THIS
print("API key configured")

API key configured


## 3. SEC Bulk 13F Data Collection

Instead of scraping individual XML filings, we download the SEC's **pre-processed quarterly bulk datasets**. Each ZIP contains clean TSV files with all 13F holdings already parsed — no XML parsing needed.

In [5]:
def download_sec_bulk_data(config):
    """Download SEC bulk 13F datasets — pre-processed TSV files."""
    headers = {"User-Agent": config.SEC_USER_AGENT, "Accept-Encoding": "gzip, deflate"}
    all_holdings = []

    for quarter in config.QUARTERS_TO_FETCH:
        url = config.SEC_BULK_URLS.get(quarter)
        if not url:
            print(f"  ⚠ No URL for {quarter}")
            continue

        print(f"\n📥 Downloading {quarter}...")
        try:
            resp = requests.get(url, headers=headers, timeout=120)
            if resp.status_code != 200:
                print(f"  ⚠ HTTP {resp.status_code} — quarter may not be published yet")
                continue

            z = zipfile.ZipFile(BytesIO(resp.content))
            print(f"  ✓ Downloaded ({len(resp.content)/1e6:.1f} MB)")

            # Find INFOTABLE and SUBMISSION files
            files = z.namelist()
            info_f = [f for f in files if "INFOTABLE" in f.upper()][0]
            sub_f = [f for f in files if "SUBMISSION" in f.upper()][0]

            infotable = pd.read_csv(z.open(info_f), sep="\t", low_memory=False)
            submissions = pd.read_csv(z.open(sub_f), sep="\t", low_memory=False)

            infotable.columns = [c.upper().strip() for c in infotable.columns]
            submissions.columns = [c.upper().strip() for c in submissions.columns]

            # Merge holdings with submission metadata
            sub_cols = ["ACCESSION_NUMBER", "CIK", "FILINGMANAGER_NAME",
                        "REPORTCALENDARORQUARTER", "FILINGDATE"]
            sub_cols = [c for c in sub_cols if c in submissions.columns]
            merged = infotable.merge(submissions[sub_cols], on="ACCESSION_NUMBER", how="left")

            # Filter to target funds
            target_set = set(config.TARGET_FUNDS.keys())
            fund_data = merged[merged["CIK"].isin(target_set)].copy()

            if len(fund_data) == 0:
                print(f"  ⚠ No target funds found")
                continue

            fund_data["fund_name"] = fund_data["CIK"].map(config.TARGET_FUNDS)
            fund_data["quarter"] = quarter

            # Standardize columns
            rename_map = {"NAMEOFISSUER": "issuer", "CUSIP": "cusip", "TITLEOFCLASS": "title",
                          "VALUE": "value", "SSHPRNAMT": "shares", "SSHPRNAMTTYPE": "share_type",
                          "INVESTMENTDISCRETION": "discretion", "CIK": "fund_cik",
                          "FILINGMANAGER_NAME": "filing_manager", "FILINGDATE": "filing_date",
                          "REPORTCALENDARORQUARTER": "report_period", "PUTCALL": "put_call"}
            for old, new in rename_map.items():
                if old in fund_data.columns:
                    fund_data.rename(columns={old: new}, inplace=True)

            for col in ["value", "shares"]:
                if col in fund_data.columns:
                    fund_data[col] = pd.to_numeric(fund_data[col], errors="coerce").fillna(0)

            print(f"  ✅ {len(fund_data):,} holdings from {fund_data['fund_name'].nunique()} funds")
            all_holdings.append(fund_data)

        except Exception as e:
            print(f"  ⚠ Error: {e}")

    if all_holdings:
        df = pd.concat(all_holdings, ignore_index=True)
        print(f"\n{'='*60}")
        print(f"Total: {len(df):,} holdings | {df['fund_name'].nunique()} funds | {df['cusip'].nunique()} securities")
        return df
    return pd.DataFrame()

holdings_df = download_sec_bulk_data(config)
holdings_df.to_csv("holdings_raw.csv", index=False)
print(f"\nSaved to holdings_raw.csv")


📥 Downloading 2025-Q4...
  ✓ Downloaded (90.3 MB)
  ✅ 27,140 holdings from 13 funds

📥 Downloading 2025-Q3...
  ✓ Downloaded (85.6 MB)
  ✅ 27,310 holdings from 14 funds

Total: 54,450 holdings | 14 funds | 8706 securities

Saved to holdings_raw.csv


## 4. Sector Classification & Data Enrichment

In [6]:
SECTOR_KEYWORDS = {
    "Technology": ["APPLE", "MICROSOFT", "GOOGLE", "ALPHABET", "META", "AMAZON",
                   "NVIDIA", "BROADCOM", "INTEL", "ADOBE", "SALESFORCE", "ORACLE",
                   "CISCO", "IBM", "QUALCOMM", "AMD", "TEXAS INSTRUMENT", "ASML",
                   "UBER", "TESLA", "PALANTIR", "SNOWFLAKE", "SERVICENOW", "INTUIT",
                   "PAYPAL", "SHOPIFY", "SNAP", "SPOTIFY", "NETFLIX"],
    "Healthcare": ["PFIZER", "JOHNSON & JOHNSON", "UNITEDHEALTH", "ELI LILLY",
                   "MERCK", "ABBVIE", "AMGEN", "GILEAD", "REGENERON", "VERTEX",
                   "MODERNA", "BIOGEN", "BRISTOL", "THERMO FISHER", "DANAHER",
                   "ABBOTT", "MEDTRONIC", "HUMANA", "CIGNA", "CVS HEALTH"],
    "Financials": ["JPMORGAN", "GOLDMAN", "MORGAN STANLEY", "BANK OF AMERICA",
                   "WELLS FARGO", "CITIGROUP", "CHARLES SCHWAB", "BLACKROCK",
                   "VISA", "MASTERCARD", "AMERICAN EXPRESS", "BERKSHIRE",
                   "PROGRESSIVE", "CAPITAL ONE", "PNC FINANCIAL", "S&P GLOBAL"],
    "Energy": ["EXXON", "CHEVRON", "CONOCOPHILLIPS", "SCHLUMBERGER", "EOG",
               "MARATHON PETROLEUM", "PIONEER", "VALERO", "OCCIDENTAL", "HESS"],
    "Consumer": ["WALMART", "COSTCO", "HOME DEPOT", "PROCTER", "COCA-COLA",
                 "PEPSICO", "MCDONALD", "NIKE", "STARBUCKS", "TARGET",
                 "CHIPOTLE", "HILTON", "MARRIOTT", "BOOKING", "AIRBNB"],
    "Industrials": ["CATERPILLAR", "DEERE", "HONEYWELL", "UNION PACIFIC",
                    "RAYTHEON", "RTX", "LOCKHEED", "BOEING", "GENERAL ELECTRIC",
                    "FEDEX", "WASTE MANAGEMENT", "CANADIAN PACIFIC"],
    "Communication": ["DISNEY", "COMCAST", "VERIZON", "AT&T", "T-MOBILE",
                      "CHARTER COMM", "WARNER", "ELECTRONIC ARTS", "ROBLOX"],
    "Utilities": ["NEXTERA", "DUKE ENERGY", "SOUTHERN CO", "DOMINION",
                  "CONSTELLATION", "EXELON", "XCEL", "SEMPRA"],
}

SECTOR_COLORS = {
    "Technology": "#4f8cff", "Healthcare": "#34d399", "Financials": "#a78bfa",
    "Energy": "#f59e42", "Consumer": "#f472b6", "Industrials": "#fbbf24",
    "Communication": "#fb923c", "Utilities": "#6ee7b7", "Other": "#6b7280",
}

def classify_sector(name):
    name_upper = str(name).upper()
    for sector, keywords in SECTOR_KEYWORDS.items():
        if any(kw in name_upper for kw in keywords):
            return sector
    return "Other"

holdings_df["sector"] = holdings_df["issuer"].apply(classify_sector)
print(f"Sector distribution:\n{holdings_df['sector'].value_counts().to_string()}")

Sector distribution:
sector
Other            51533
Technology         993
Financials         414
Healthcare         372
Consumer           340
Utilities          236
Energy             209
Communication      200
Industrials        153


## 5. Quarter-over-Quarter Change Detection

In [7]:
def compute_changes(df):
    if df.empty or df["quarter"].nunique() < 2:
        print("Need 2+ quarters for change detection")
        df["change_type"] = "held"
        df["shares_change"] = 0
        df["shares_change_pct"] = 0.0
        df["value_change"] = 0
        return df

    quarters = sorted(df["quarter"].unique())
    changes = []

    for fund in df["fund_name"].unique():
        fd = df[df["fund_name"] == fund]
        for i in range(1, len(quarters)):
            prev_q, curr_q = quarters[i-1], quarters[i]
            prev = fd[fd["quarter"] == prev_q].drop_duplicates("cusip").set_index("cusip")
            curr = fd[fd["quarter"] == curr_q].drop_duplicates("cusip").set_index("cusip")

            for cusip in set(prev.index) | set(curr.index):
                in_p, in_c = cusip in prev.index, cusip in curr.index

                if in_c:
                    r = curr.loc[cusip]
                    if isinstance(r, pd.DataFrame): r = r.iloc[0]
                    rec = r.to_dict()
                else:
                    r = prev.loc[cusip]
                    if isinstance(r, pd.DataFrame): r = r.iloc[0]
                    rec = r.to_dict()
                    rec["quarter"] = curr_q

                if in_c and not in_p:
                    rec.update({"change_type": "new_position", "shares_change": rec.get("shares",0),
                                "shares_change_pct": 100.0, "value_change": rec.get("value",0)})
                elif in_p and not in_c:
                    pr = prev.loc[cusip]
                    if isinstance(pr, pd.DataFrame): pr = pr.iloc[0]
                    rec.update({"change_type": "exit", "shares_change": -pr.get("shares",0),
                                "shares_change_pct": -100.0, "value_change": -pr.get("value",0)})
                else:
                    pr = prev.loc[cusip]
                    if isinstance(pr, pd.DataFrame): pr = pr.iloc[0]
                    ps, cs = pr.get("shares",0), rec.get("shares",0)
                    pct = ((cs - ps) / ps * 100) if ps > 0 else 0.0
                    rec["shares_change"] = cs - ps
                    rec["shares_change_pct"] = pct
                    rec["value_change"] = rec.get("value",0) - pr.get("value",0)
                    if pct > 5: rec["change_type"] = "increased"
                    elif pct < -5: rec["change_type"] = "decreased"
                    else: rec["change_type"] = "held"

                rec["comparison_quarter"] = f"{prev_q} → {curr_q}"
                changes.append(rec)

    result = pd.DataFrame(changes)
    if "sector" not in result.columns and "issuer" in result.columns:
        result["sector"] = result["issuer"].apply(classify_sector)
    print(f"Change detection:\n{result['change_type'].value_counts().to_string()}")
    return result

changes_df = compute_changes(holdings_df)

Change detection:
change_type
decreased       6167
increased       5863
new_position    2817
exit            2733
held            1696


## 6. Financial Metrics & KPIs

Compute fund-level and position-level metrics: portfolio concentration (HHI), estimated returns, win rate, turnover, and position-level PnL.

In [8]:
def compute_fund_kpis(holdings_df, changes_df):
    """Compute rich KPIs for each fund."""
    latest_q = holdings_df["quarter"].max()
    current = holdings_df[holdings_df["quarter"] == latest_q]
    fund_kpis = []

    for fund in current["fund_name"].unique():
        fd = current[current["fund_name"] == fund].copy()
        total_val = fd["value"].sum()
        if total_val == 0:
            continue

        fd["weight"] = fd["value"] / total_val * 100

        # Concentration metrics
        hhi = (fd["weight"] ** 2).sum()  # Herfindahl-Hirschman Index
        top1_w = fd["weight"].max()
        top5_w = fd.nlargest(5, "weight")["weight"].sum()
        top10_w = fd.nlargest(10, "weight")["weight"].sum()
        top1_name = fd.loc[fd["value"].idxmax(), "issuer"]

        # Sector weights
        sector_w = fd.groupby("sector")["weight"].sum().to_dict()
        max_sector = max(sector_w, key=sector_w.get) if sector_w else "N/A"

        # Change-based metrics (if changes available)
        fc = changes_df[changes_df["fund_name"] == fund] if not changes_df.empty else pd.DataFrame()
        num_new = len(fc[fc["change_type"] == "new_position"]) if "change_type" in fc.columns else 0
        num_exits = len(fc[fc["change_type"] == "exit"]) if "change_type" in fc.columns else 0
        num_increased = len(fc[fc["change_type"] == "increased"]) if "change_type" in fc.columns else 0
        num_decreased = len(fc[fc["change_type"] == "decreased"]) if "change_type" in fc.columns else 0

        # Estimated PnL from value changes
        if "value_change" in fc.columns and len(fc) > 0:
            total_pnl = fc["value_change"].sum()
            winners = fc[fc["value_change"] > 0]
            losers = fc[fc["value_change"] < 0]
            win_rate = len(winners) / len(fc) * 100 if len(fc) > 0 else 0
            best_pos = winners.nlargest(1, "value_change").iloc[0] if len(winners) > 0 else None
            worst_pos = losers.nsmallest(1, "value_change").iloc[0] if len(losers) > 0 else None
        else:
            total_pnl, win_rate = 0, 0
            best_pos, worst_pos = None, None

        # Turnover (new + exits) / total positions
        turnover = (num_new + num_exits) / len(fd) * 100 if len(fd) > 0 else 0

        fund_kpis.append({
            "fund_name": fund, "total_value": total_val, "num_positions": len(fd),
            "hhi": round(hhi, 1), "top1_weight": round(top1_w, 1),
            "top5_weight": round(top5_w, 1), "top10_weight": round(top10_w, 1),
            "top1_holding": top1_name, "max_sector": max_sector,
            "max_sector_weight": round(sector_w.get(max_sector, 0), 1),
            "num_new": num_new, "num_exits": num_exits,
            "num_increased": num_increased, "num_decreased": num_decreased,
            "turnover_pct": round(turnover, 1),
            "estimated_pnl": total_pnl, "win_rate": round(win_rate, 1),
            "best_position": best_pos["issuer"] if best_pos is not None else "N/A",
            "best_pnl": best_pos["value_change"] if best_pos is not None else 0,
            "worst_position": worst_pos["issuer"] if worst_pos is not None else "N/A",
            "worst_pnl": worst_pos["value_change"] if worst_pos is not None else 0,
            "sector_weights": sector_w,
        })

    return pd.DataFrame(fund_kpis)

kpi_df = compute_fund_kpis(holdings_df, changes_df)
print(f"Computed KPIs for {len(kpi_df)} funds")
display(kpi_df[["fund_name", "total_value", "num_positions", "hhi",
                "top1_weight", "turnover_pct", "win_rate"]].head(10))

Computed KPIs for 13 funds


,fund_name,total_value,num_positions,hhi,top1_weight,turnover_pct,win_rate
0,Baupost Group,3984029993,8,2748.4,38.1,0.0,57.1
1,Appaloosa Management,7274621868,44,427.2,7.6,52.3,44.6
2,Icahn Capital,8445167769,19,1322.2,24.6,0.0,61.5
3,Citadel Advisors,665872168045,15403,73.3,3.3,15.9,49.5
4,Bridgewater Associates,15526737802,11,1299.1,18.1,18.2,50.0
5,Viking Global Investors,237791015622,5950,60.4,4.2,16.2,47.6
6,Tiger Global Management,27387619221,1719,304.5,16.7,16.5,64.5
7,Berkshire Hathaway (Buffett),274160086701,110,701.8,20.1,6.4,46.7
8,Two Sigma Investments,29714313270,54,521.9,11.2,7.4,33.3
9,Coatue Management,1227166306,371,323.8,8.0,100.5,75.3


## 7. Interactive Dashboard

Rich visualisations with fund-level KPIs, PnL tracking, concentration analysis, sector flows, and cross-fund conviction signals.

In [9]:
# ================================================================
# 7a. OVERVIEW METRICS BANNER
# ================================================================
latest_q = holdings_df["quarter"].max()
current = holdings_df[holdings_df["quarter"] == latest_q]
tv = current["value"].sum()
nf = current["fund_name"].nunique()
np_ = len(current)
ns = current["cusip"].nunique()
nn = len(changes_df[changes_df["change_type"]=="new_position"]) if "change_type" in changes_df.columns else 0
ne = len(changes_df[changes_df["change_type"]=="exit"]) if "change_type" in changes_df.columns else 0

display(HTML(f"""
<div style="background:#0c0e14; padding:24px; border-radius:16px; margin:10px 0 20px 0;">
<h1 style="color:#e8eaf0; font-family:sans-serif; margin:0 0 4px 0;">🏦 Institutional Flow Tracker</h1>
<p style="color:#5c6380; font-family:sans-serif; margin:0 0 20px 0;">SEC 13F · {latest_q} · {nf} Funds</p>
<div style="display:flex; gap:12px; flex-wrap:wrap;">
<div style="background:#13151e; border:1px solid #1e2130; border-radius:12px; padding:16px 22px; text-align:center; flex:1; min-width:110px;">
<div style="font-size:28px; font-weight:800; color:#4f8cff; font-family:monospace;">{nf}</div>
<div style="font-size:11px; color:#5c6380; text-transform:uppercase;">Funds</div></div>
<div style="background:#13151e; border:1px solid #1e2130; border-radius:12px; padding:16px 22px; text-align:center; flex:1; min-width:110px;">
<div style="font-size:28px; font-weight:800; color:#34d399; font-family:monospace;">{np_:,}</div>
<div style="font-size:11px; color:#5c6380; text-transform:uppercase;">Positions</div></div>
<div style="background:#13151e; border:1px solid #1e2130; border-radius:12px; padding:16px 22px; text-align:center; flex:1; min-width:110px;">
<div style="font-size:28px; font-weight:800; color:#f59e42; font-family:monospace;">{ns:,}</div>
<div style="font-size:11px; color:#5c6380; text-transform:uppercase;">Securities</div></div>
<div style="background:#13151e; border:1px solid #1e2130; border-radius:12px; padding:16px 22px; text-align:center; flex:1; min-width:110px;">
<div style="font-size:28px; font-weight:800; color:#a78bfa; font-family:monospace;">${tv/1e9:.1f}B</div>
<div style="font-size:11px; color:#5c6380; text-transform:uppercase;">AUM</div></div>
<div style="background:#13151e; border:1px solid #1e2130; border-radius:12px; padding:16px 22px; text-align:center; flex:1; min-width:110px;">
<div style="font-size:28px; font-weight:800; color:#34d399; font-family:monospace;">{nn}</div>
<div style="font-size:11px; color:#5c6380; text-transform:uppercase;">New Positions</div></div>
<div style="background:#13151e; border:1px solid #1e2130; border-radius:12px; padding:16px 22px; text-align:center; flex:1; min-width:110px;">
<div style="font-size:28px; font-weight:800; color:#f87171; font-family:monospace;">{ne}</div>
<div style="font-size:11px; color:#5c6380; text-transform:uppercase;">Exits</div></div>
</div></div>"""))

In [10]:
# ================================================================
# 7b. FUND KPI SCORECARDS
# ================================================================
fig = make_subplots(rows=2, cols=3, subplot_titles=[
    "Portfolio Size ($B)", "Portfolio Concentration (HHI)",
    "Win Rate (%)", "Turnover (%)", "Top-1 Holding Weight (%)", "Estimated PnL ($M)"
])

kpi_sorted = kpi_df.sort_values("total_value", ascending=True)
short_names = kpi_sorted["fund_name"].str.split("(").str[0].str.strip()

fig.add_trace(go.Bar(y=short_names, x=kpi_sorted["total_value"]/1e9, orientation="h",
    marker_color="#4f8cff"), row=1, col=1)
fig.add_trace(go.Bar(y=short_names, x=kpi_sorted["hhi"], orientation="h",
    marker_color=["#f87171" if h > 1000 else "#34d399" for h in kpi_sorted["hhi"]]), row=1, col=2)
fig.add_trace(go.Bar(y=short_names, x=kpi_sorted["win_rate"], orientation="h",
    marker_color=["#34d399" if w >= 50 else "#f87171" for w in kpi_sorted["win_rate"]]), row=1, col=3)
fig.add_trace(go.Bar(y=short_names, x=kpi_sorted["turnover_pct"], orientation="h",
    marker_color="#f59e42"), row=2, col=1)
fig.add_trace(go.Bar(y=short_names, x=kpi_sorted["top1_weight"], orientation="h",
    marker_color="#a78bfa"), row=2, col=2)
fig.add_trace(go.Bar(y=short_names, x=kpi_sorted["estimated_pnl"]/1e6, orientation="h",
    marker_color=["#34d399" if p > 0 else "#f87171" for p in kpi_sorted["estimated_pnl"]]), row=2, col=3)

fig.update_layout(template="plotly_dark", height=900, showlegend=False,
    paper_bgcolor="#0c0e14", plot_bgcolor="#13151e",
    title="<b>Fund KPI Dashboard</b>", font=dict(family="sans-serif"))
fig.show()

In [11]:
# ================================================================
# 7c. TOP HOLDINGS & SECTOR ALLOCATION
# ================================================================
# Top 25 holdings
top = current.groupby(["cusip","issuer","sector"]).agg(
    total_value=("value","sum"), num_funds=("fund_name","nunique")
).reset_index().nlargest(25, "total_value").sort_values("total_value")
top["value_b"] = top["total_value"] / 1e9
top["label"] = top["issuer"] + " (" + top["num_funds"].astype(str) + " funds)"

fig1 = px.bar(top, x="value_b", y="label", orientation="h", color="sector",
    color_discrete_map=SECTOR_COLORS, title="<b>Top 25 Holdings by Aggregate Value</b>",
    labels={"value_b": "Value ($B)", "label": ""})
fig1.update_layout(template="plotly_dark", height=700, paper_bgcolor="#0c0e14",
    plot_bgcolor="#13151e", legend=dict(orientation="h", y=-0.1))
fig1.show()

# Sector breakdown
sec_agg = current.groupby("sector")["value"].sum().reset_index().sort_values("value", ascending=False)
sec_agg["pct"] = (sec_agg["value"] / sec_agg["value"].sum() * 100).round(1)
colors = [SECTOR_COLORS.get(s, "#6b7280") for s in sec_agg["sector"]]

fig2 = go.Figure(go.Pie(labels=sec_agg["sector"], values=sec_agg["value"],
    marker=dict(colors=colors), hole=0.45, textinfo="label+percent"))
fig2.update_layout(template="plotly_dark", height=450, paper_bgcolor="#0c0e14",
    title="<b>Aggregate Sector Allocation</b>")
fig2.show()

In [12]:
# ================================================================
# 7d. SECTOR HEATMAP PER FUND
# ================================================================
fs = current.groupby(["fund_name","sector"])["value"].sum().reset_index()
ft = current.groupby("fund_name")["value"].sum()
fs["pct"] = fs.apply(lambda r: r["value"]/ft[r["fund_name"]]*100 if ft[r["fund_name"]]>0 else 0, axis=1).round(1)
pivot = fs.pivot_table(index="fund_name", columns="sector", values="pct", fill_value=0)

fig3 = px.imshow(pivot, color_continuous_scale="blues", aspect="auto",
    title="<b>Sector Allocation by Fund (%)</b>", labels=dict(color="% Portfolio"))
fig3.update_layout(template="plotly_dark", height=600, paper_bgcolor="#0c0e14",
    xaxis=dict(tickangle=-45))
fig3.show()

In [13]:
# ================================================================
# 7e. PORTFOLIO CONCENTRATION
# ================================================================
conc = kpi_df.sort_values("top1_weight", ascending=True)
short = conc["fund_name"].str.split("(").str[0].str.strip()

fig4 = go.Figure()
fig4.add_trace(go.Bar(y=short, x=conc["top1_weight"], name="Top 1", orientation="h",
    marker_color="#f87171", text=conc["top1_holding"], textposition="inside", textfont=dict(size=9)))
fig4.add_trace(go.Bar(y=short, x=conc["top5_weight"]-conc["top1_weight"], name="Pos 2-5",
    orientation="h", marker_color="#f59e42"))
fig4.add_trace(go.Bar(y=short, x=conc["top10_weight"]-conc["top5_weight"], name="Pos 6-10",
    orientation="h", marker_color="#4f8cff"))
fig4.add_trace(go.Bar(y=short, x=100-conc["top10_weight"], name="Rest",
    orientation="h", marker_color="#1e2130"))
fig4.update_layout(barmode="stack", template="plotly_dark", height=650,
    paper_bgcolor="#0c0e14", plot_bgcolor="#13151e",
    title="<b>Portfolio Concentration</b>", xaxis=dict(title="% of Portfolio", range=[0,100]),
    legend=dict(orientation="h", y=-0.1))
fig4.show()

In [14]:
# ================================================================
# 7f. POSITION CHANGE SIGNALS & PnL
# ================================================================
if not changes_df.empty and "change_type" in changes_df.columns:
    # New positions
    new_pos = changes_df[changes_df["change_type"]=="new_position"].copy()
    if len(new_pos) > 0:
        nt = new_pos.nlargest(15, "value")
        nt["value_m"] = nt["value"] / 1e6
        nt["label"] = nt["fund_name"].str.split("(").str[0].str.strip() + " → " + nt["issuer"]
        fig5 = px.bar(nt.sort_values("value_m"), x="value_m", y="label", orientation="h",
            color="sector", color_discrete_map=SECTOR_COLORS,
            title="<b>🟢 Largest New Positions</b>", labels={"value_m": "Value ($M)", "label": ""})
        fig5.update_layout(template="plotly_dark", height=500, paper_bgcolor="#0c0e14",
            plot_bgcolor="#13151e", legend=dict(orientation="h", y=-0.15))
        fig5.show()

    # Exits
    exits = changes_df[changes_df["change_type"]=="exit"].copy()
    if len(exits) > 0:
        exits["abs_val"] = exits["value_change"].abs()
        et = exits.nlargest(15, "abs_val")
        et["value_m"] = et["abs_val"] / 1e6
        et["label"] = et["fund_name"].str.split("(").str[0].str.strip() + " ✕ " + et["issuer"]
        fig6 = px.bar(et.sort_values("value_m"), x="value_m", y="label", orientation="h",
            color="sector", color_discrete_map=SECTOR_COLORS,
            title="<b>🔴 Largest Exits</b>", labels={"value_m": "Exited Value ($M)", "label": ""})
        fig6.update_layout(template="plotly_dark", height=500, paper_bgcolor="#0c0e14",
            plot_bgcolor="#13151e", legend=dict(orientation="h", y=-0.15))
        fig6.show()

    # PnL waterfall per fund
    pnl = kpi_df[["fund_name","estimated_pnl"]].sort_values("estimated_pnl")
    pnl["pnl_m"] = pnl["estimated_pnl"] / 1e6
    pnl["short"] = pnl["fund_name"].str.split("(").str[0].str.strip()
    pnl["color"] = pnl["estimated_pnl"].apply(lambda x: "#34d399" if x > 0 else "#f87171")

    fig7 = go.Figure(go.Bar(y=pnl["short"], x=pnl["pnl_m"], orientation="h",
        marker_color=pnl["color"]))
    fig7.update_layout(template="plotly_dark", height=600, paper_bgcolor="#0c0e14",
        plot_bgcolor="#13151e", title="<b>Estimated PnL by Fund ($M)</b>",
        xaxis=dict(title="Estimated PnL ($M)"))
    fig7.show()

In [15]:
# ================================================================
# 7g. CROSS-FUND CONVICTION & TREEMAP
# ================================================================
overlap = current.groupby(["cusip","issuer","sector"]).agg(
    num_funds=("fund_name","nunique"), total_value=("value","sum"),
    funds=("fund_name", lambda x: list(sorted(set(x))))
).reset_index().sort_values("num_funds", ascending=False)

multi = overlap[overlap["num_funds"] >= 3].head(20).sort_values("num_funds")
if len(multi) > 0:
    fig8 = px.bar(multi, x="num_funds", y="issuer", orientation="h",
        color="sector", color_discrete_map=SECTOR_COLORS,
        title="<b>Cross-Fund Conviction</b> — Stocks held by 3+ funds",
        labels={"num_funds": "# Funds", "issuer": ""})
    fig8.update_layout(template="plotly_dark", height=550, paper_bgcolor="#0c0e14",
        plot_bgcolor="#13151e", legend=dict(orientation="h", y=-0.15))
    fig8.show()

# Treemap
tree = current.groupby(["fund_name","issuer","sector"])["value"].sum().reset_index()
tree = tree.sort_values("value", ascending=False).groupby("fund_name").head(8)
fig9 = px.treemap(tree, path=["fund_name","sector","issuer"], values="value",
    color="sector", color_discrete_map=SECTOR_COLORS,
    title="<b>Holdings Treemap</b> — Top 8 per fund")
fig9.update_layout(template="plotly_dark", height=650, paper_bgcolor="#0c0e14",
    margin=dict(l=10,r=10,t=50,b=10))
fig9.show()

display(HTML('<div style="background:#13151e; border:1px solid #1e2130; border-radius:12px; padding:14px; margin:16px 0; text-align:center;"><span style="color:#34d399; font-weight:700;">✅ Dashboard Complete</span> <span style="color:#5c6380; margin-left:10px;">All charts interactive — hover, zoom, click legends</span></div>'))

## 8. Document Construction & Chunking

**Baseline**: Naive fixed-size chunking
**Enhanced**: Hierarchical chunking into 4 types: `fund_overview`, `position_detail`, `change_alert`, `cross_fund_holding`

In [29]:
@dataclass
class Document:
    doc_id: str
    text: str
    doc_type: str
    metadata: Dict = field(default_factory=dict)

def build_enhanced_documents(holdings_df, changes_df):
    docs = []

    # ONLY use the latest quarter per fund (current holdings only)
    latest_per_fund = holdings_df.groupby("fund_name")["quarter"].transform("max")
    current = holdings_df[holdings_df["quarter"] == latest_per_fund].copy()
    print(f"Filtered to current holdings: {len(holdings_df):,} \u2192 {len(current):,} positions")

    # Fund overviews
    for fund in current["fund_name"].unique():
        fd = current[current["fund_name"] == fund]
        tv = fd["value"].sum()
        top5 = fd.nlargest(5, "value")
        top_txt = "\n".join([f"  - {r['issuer']} ({r['cusip']}): ${r['value']:,d} ({r['shares']:,d} shares)"
                              for _, r in top5.iterrows()])
        txt = f"Fund Overview: {fund}\nQuarter: {fd['quarter'].iloc[0]}\nTotal Value: ${tv:,d}\n# Positions: {len(fd)}\nTop 5:\n{top_txt}"
        docs.append(Document(f"overview_{fund[:20]}_{fd['quarter'].iloc[0]}", txt, "fund_overview",
            {"fund_name": fund, "quarter": fd['quarter'].iloc[0], "total_value": int(tv)}))

    # Position details \u2014 use enumerate for unique IDs
    for idx, (_, r) in enumerate(current.iterrows()):
        txt = (f"Position: {r.get('fund_name','?')} holds {r.get('issuer','?')} ({r.get('cusip','')}).\n"
               f"Shares: {r.get('shares',0):,.0f} | Value: ${r.get('value',0):,.0f} | Sector: {r.get('sector','?')}")
        docs.append(Document(f"pos_{idx}", txt, "position_detail",
            {"fund_name": str(r.get("fund_name","")), "issuer": str(r.get("issuer","")),
             "cusip": str(r.get("cusip","")), "value": int(r.get("value",0)),
             "sector": str(r.get("sector",""))}))

    # Change alerts \u2014 use enumerate for unique IDs
    if not changes_df.empty and "change_type" in changes_df.columns:
        active = changes_df[changes_df["change_type"].isin(["new_position","exit","increased","decreased"])]
        for idx, (_, r) in enumerate(active.iterrows()):
            ct = r["change_type"]
            fn = r.get("fund_name", "?")
            iss = r.get("issuer", "?")
            cus = r.get("cusip", "")

            if ct == "new_position":
                txt = f"NEW POSITION: {fn} bought {iss} ({cus}).\nShares: {r.get('shares',0):,.0f} | Value: ${r.get('value',0):,.0f}"
            elif ct == "exit":
                txt = f"EXIT: {fn} sold all {iss} ({cus}).\nPrev value: ${abs(r.get('value_change',0)):,.0f}"
            elif ct == "increased":
                txt = f"INCREASED: {fn} added to {iss} ({cus}).\nChange: +{r.get('shares_change_pct',0):.1f}% | Value: ${r.get('value',0):,.0f}"
            else:
                txt = f"DECREASED: {fn} trimmed {iss} ({cus}).\nChange: {r.get('shares_change_pct',0):.1f}% | Value: ${r.get('value',0):,.0f}"

            docs.append(Document(f"chg_{idx}", txt, "change_alert",
                {"fund_name": str(fn), "issuer": str(iss), "change_type": str(ct), "cusip": str(cus)}))

    # Cross-fund holdings
    stock_funds = current.groupby("cusip").agg(
        fund_name=("fund_name", list), issuer=("issuer", "first"),
        value=("value", "sum"), shares=("shares", "sum")
    ).reset_index()

    for idx, (_, r) in enumerate(stock_funds[stock_funds["fund_name"].apply(len) >= 2].iterrows()):
        funds_str = ", ".join(r["fund_name"])
        txt = f"Cross-Fund: {r['issuer']} ({r['cusip']}) held by {len(r['fund_name'])} funds.\nFunds: {funds_str}\nCombined: ${r['value']:,d}"
        docs.append(Document(f"cross_{idx}", txt, "cross_fund_holding",
            {"issuer": str(r["issuer"]), "cusip": str(r["cusip"]), "num_funds": int(len(r["fund_name"]))}))

    types = defaultdict(int)
    for d in docs: types[d.doc_type] += 1
    print(f"Built {len(docs)} documents: {dict(types)}")
    return docs

def naive_chunk(docs, size=512, overlap=50):
    chunked = []
    for d in docs:
        if len(d.text) <= size:
            chunked.append(d)
        else:
            s = 0
            i = 0
            while s < len(d.text):
                chunked.append(Document(f"{d.doc_id}_c{i}", d.text[s:s+size], d.doc_type,
                    {**d.metadata, "chunk_idx": i}))
                s += size - overlap
                i += 1
    print(f"Naive chunking: {len(docs)} \u2192 {len(chunked)} chunks")
    return chunked

enhanced_docs = build_enhanced_documents(holdings_df, changes_df)
baseline_docs = naive_chunk(enhanced_docs)

Filtered to current holdings: 54,450 → 27,148 positions
Built 49732 documents: {'fund_overview': 14, 'position_detail': 27148, 'change_alert': 17580, 'cross_fund_holding': 4990}
Naive chunking: 49732 → 49736 chunks


## 9. Embedding, Retrieval & Reranking

In [30]:
class VectorStore:
    def __init__(self, name, model_name):
        self.model = SentenceTransformer(model_name)
        self.client = chromadb.Client()
        self.col = self.client.get_or_create_collection(name, metadata={"hnsw:space":"cosine"})
    def add(self, docs, bs=100):
        for i in tqdm(range(0,len(docs),bs), desc="Embedding"):
            b = docs[i:i+bs]
            emb = self.model.encode([d.text for d in b], show_progress_bar=False).tolist()
            self.col.upsert(ids=[d.doc_id for d in b], embeddings=emb, documents=[d.text for d in b],
                metadatas=[{k: int(v) if isinstance(v, (np.integer,)) else float(v) if isinstance(v, (np.floating,)) else str(v) if isinstance(v, (list, dict)) else v for k, v in d.metadata.items()} for d in b]
)
    def query(self, q, k=10):
        r = self.col.query(query_embeddings=self.model.encode([q]).tolist(), n_results=k,
            include=["documents","metadatas","distances"])
        return [{"doc_id":r["ids"][0][i], "text":r["documents"][0][i],
                 "metadata":r["metadatas"][0][i] if r["metadatas"] else {}, "score":1-r["distances"][0][i]}
                for i in range(len(r["ids"][0]))]

class BM25Index:
    def __init__(self, docs):
        self.docs = docs
        self.bm25 = BM25Okapi([d.text.lower().split() for d in docs])
    def query(self, q, k=10):
        scores = self.bm25.get_scores(q.lower().split())
        top = np.argsort(scores)[::-1][:k]
        return [{"doc_id":self.docs[i].doc_id, "text":self.docs[i].text,
                 "metadata":self.docs[i].metadata, "score":float(scores[i])} for i in top if scores[i]>0]

class Reranker:
    def __init__(self, name):
        self.model = CrossEncoder(name)
    def rerank(self, q, docs, k=8):
        if not docs: return []
        scores = self.model.predict([[q, d["text"]] for d in docs])
        for i, d in enumerate(docs): d["rerank_score"] = float(scores[i])
        return sorted(docs, key=lambda x: x["rerank_score"], reverse=True)[:k]

class BaselineRetriever:
    def __init__(self, vs, k=8): self.vs, self.k = vs, k
    def retrieve(self, q): return self.vs.query(q, self.k)

class EnhancedRetriever:
    def __init__(self, vs, bm25, reranker, cfg):
        self.vs, self.bm25, self.reranker, self.cfg = vs, bm25, reranker, cfg
    def retrieve(self, q):
        vr = self.vs.query(q, self.cfg.TOP_K_RETRIEVAL)
        br = self.bm25.query(q, self.cfg.TOP_K_RETRIEVAL)
        fused = self._rrf(vr, br)
        return self.reranker.rerank(q, fused, self.cfg.TOP_K_RERANK)
    def _rrf(self, vr, br, k=60):
        scores, data = {}, {}
        for r, d in enumerate(vr):
            scores[d["doc_id"]] = scores.get(d["doc_id"],0) + self.cfg.VECTOR_WEIGHT/(k+r+1)
            data[d["doc_id"]] = d
        for r, d in enumerate(br):
            scores[d["doc_id"]] = scores.get(d["doc_id"],0) + self.cfg.BM25_WEIGHT/(k+r+1)
            if d["doc_id"] not in data: data[d["doc_id"]] = d
        return [{**data[did], "rrf_score": scores[did]}
                for did in sorted(scores, key=scores.get, reverse=True)[:self.cfg.TOP_K_RETRIEVAL]]

# Build everything
print("Building vector stores...")
bs = VectorStore("baseline", config.EMBEDDING_MODEL)
bs.add(baseline_docs)
es = VectorStore("enhanced", config.EMBEDDING_MODEL)
es.add(enhanced_docs)
bm25 = BM25Index(enhanced_docs)
reranker = Reranker(config.RERANKER_MODEL)
baseline_retriever = BaselineRetriever(bs, config.TOP_K_BASELINE)
enhanced_retriever = EnhancedRetriever(es, bm25, reranker, config)
print("Both pipelines ready")

Building vector stores...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Embedding: 100%|██████████| 498/498 [18:31<00:00,  2.23s/it]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Embedding: 100%|██████████| 498/498 [18:28<00:00,  2.23s/it]


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Both pipelines ready


## 10. Generation (Groq — Llama 3.3 70B)

In [31]:
class RAGGenerator:
    def __init__(self, cfg):
        self.cfg = cfg
        self.client = Groq(api_key=cfg.GROQ_API_KEY)

    def generate_baseline(self, q, retrieved):
        ctx = "\n\n".join([d["text"] for d in retrieved])
        prompt = f"Answer based on this context:\n\n{ctx}\n\nQuestion: {q}\nAnswer:"
        return {"query":q, "answer":self._llm(prompt), "sources":[d.get("doc_id","") for d in retrieved],
                "pipeline":"baseline", "num_sources":len(retrieved), "retrieved_docs":retrieved}

    def generate_enhanced(self, q, retrieved):
        parts = []
        for i, d in enumerate(retrieved):
            dt = d.get("metadata",{}).get("doc_type","?")
            fn = d.get("metadata",{}).get("fund_name","?")
            parts.append(f"[Source {i+1}] ({dt}, {fn})\n{d['text']}")
        ctx = "\n\n---\n\n".join(parts)
        sys = ("You are an institutional investment analyst. Answer ONLY from the provided sources. "
               "Cite [Source N]. Specify quarters and percentages. Flag limitations.")
        prompt = f"Query: {q}\n\nSources:\n{ctx}\n\nGrounded answer with citations:"
        return {"query":q, "answer":self._llm(prompt, sys), "sources":[d.get("doc_id","") for d in retrieved],
                "rerank_scores":[d.get("rerank_score",0) for d in retrieved],
                "pipeline":"enhanced", "num_sources":len(retrieved), "retrieved_docs":retrieved}

    def _llm(self, prompt, system=None):
        for attempt in range(3):
            try:
                time.sleep(2)
                msgs = []
                if system: msgs.append({"role":"system","content":system})
                msgs.append({"role":"user","content":prompt})
                r = self.client.chat.completions.create(model=self.cfg.LLM_MODEL, messages=msgs,
                    temperature=self.cfg.LLM_TEMPERATURE, max_tokens=self.cfg.LLM_MAX_TOKENS)
                return r.choices[0].message.content.strip()
            except Exception as e:
                if "429" in str(e):
                    wait = 15*(attempt+1)
                    print(f"  ⏳ Rate limited, waiting {wait}s...")
                    time.sleep(wait)
                else:
                    return f"[LLM Error: {e}]"
        return "[Rate limit exceeded after 3 retries]"

generator = RAGGenerator(config)

class RAGPipeline:
    def __init__(self, br, er, gen):
        self.br, self.er, self.gen = br, er, gen
    def query_baseline(self, q): return self.gen.generate_baseline(q, self.br.retrieve(q))
    def query_enhanced(self, q): return self.gen.generate_enhanced(q, self.er.retrieve(q))
    def compare(self, q):
        print(f"\n{'='*60}\nQUERY: {q}\n{'='*60}")
        b, e = self.query_baseline(q), self.query_enhanced(q)
        print(f"\n--- BASELINE ---\n{b['answer'][:300]}")
        print(f"\n--- ENHANCED ---\n{e['answer'][:300]}")
        return {"query":q, "baseline":b, "enhanced":e}

pipeline = RAGPipeline(baseline_retriever, enhanced_retriever, generator)
print("Pipeline ready")

Pipeline ready


## 11. Interactive Query

In [32]:
QUERY = "What new positions did Citadel Advisors initiate?"
result = pipeline.query_enhanced(QUERY)
display(Markdown(f"### RAG Response\n\n{result['answer']}"))
print(f"\nSources: {result['num_sources']}")

### RAG Response

Based on the provided sources, it appears that Citadel Advisors initiated new positions in the following companies: 

1. NEWS CORP NEW (65249B208) [Source 5, Source 6] with a maximum of 31,489 shares and a value of $933,019 [Source 5].
2. ADVISOR MANAGED PORTFOLIOS, with two different holdings: 
   - (00777X660) [Source 7] with 7,409 shares and a value of $346,411.
   - (00777X538) [Source 8] with 8,437 shares and a value of $304,072.

However, the sources do not provide information on the specific quarter or percentage of the portfolio allocated to these new positions. 

Flagged limitations: 
- The sources do not provide a clear timeline or quarter for the initiation of these new positions.
- The sources do not provide information on the percentage of the portfolio allocated to these new positions.
- The sources only provide information on the number of shares and value of the holdings, but not on the overall portfolio size or composition. [Source 1-8]


Sources: 8


## 12. Evaluation

In [33]:
TEST_QUERIES = [
    {"query":"What are Berkshire Hathaway's top 5 holdings?", "type":"factual",
     "expected_doc_types":["fund_overview","position_detail"], "expected_funds":["Berkshire Hathaway (Buffett)"], "difficulty":"easy"},
    {"query":"What new positions did Citadel initiate?", "type":"change",
     "expected_doc_types":["change_alert"], "expected_funds":["Citadel Advisors"], "difficulty":"medium"},
    {"query":"Which stocks are held by the most funds?", "type":"cross_fund",
     "expected_doc_types":["cross_fund_holding"], "expected_funds":[], "difficulty":"medium"},
    {"query":"Are institutions increasing tech exposure?", "type":"sector_trend",
     "expected_doc_types":["change_alert","fund_overview"], "expected_funds":[], "difficulty":"hard"},
    {"query":"Which fund has the most concentrated portfolio?", "type":"analysis",
     "expected_doc_types":["fund_overview"], "expected_funds":[], "difficulty":"hard"},
    {"query":"Did any fund exit Apple recently?", "type":"edge_case",
     "expected_doc_types":["change_alert"], "expected_funds":[], "difficulty":"medium"},
    {"query":"Compare Bridgewater and Renaissance strategies", "type":"comparison",
     "expected_doc_types":["fund_overview","position_detail"],
     "expected_funds":["Bridgewater Associates","Renaissance Technologies"], "difficulty":"hard"},
    {"query":"Total institutional money in energy stocks?", "type":"aggregation",
     "expected_doc_types":["position_detail","cross_fund_holding"], "expected_funds":[], "difficulty":"hard"},
    {"query":"Show position changes >50%", "type":"threshold",
     "expected_doc_types":["change_alert"], "expected_funds":[], "difficulty":"medium"},
    {"query":"Which fund is most bullish on healthcare?", "type":"sentiment",
     "expected_doc_types":["change_alert","fund_overview"], "expected_funds":[], "difficulty":"hard"},
]

def eval_retrieval(queries, retriever, name):
    results = {"precision":[], "recall":[], "mrr":[]}
    for tq in tqdm(queries, desc=name):
        ret = retriever.retrieve(tq["query"])
        exp_types = set(tq["expected_doc_types"])
        exp_funds = set(tq.get("expected_funds",[]))
        rel, first = 0, None
        for rank, d in enumerate(ret):
            m = d.get("metadata",{})
            if m.get("doc_type","") in exp_types or (exp_funds and m.get("fund_name","") in exp_funds):
                rel += 1
                if first is None: first = rank + 1
        k = len(ret)
        results["precision"].append(rel/k if k>0 else 0)
        results["recall"].append(min(rel/max(len(exp_types),1), 1.0))
        results["mrr"].append(1.0/first if first else 0)
    for m in ["precision","recall","mrr"]:
        results[f"avg_{m}"] = np.mean(results[m])
    print(f"{name}: P@K={results['avg_precision']:.3f} R@K={results['avg_recall']:.3f} MRR={results['avg_mrr']:.3f}")
    return results

def eval_generation(queries, pipeline, enhanced=True):
    results = []
    label = "Enhanced" if enhanced else "Baseline"
    for tq in tqdm(queries, desc=f"Gen {label}"):
        r = pipeline.query_enhanced(tq["query"]) if enhanced else pipeline.query_baseline(tq["query"])
        ep = f'Rate this answer. Query: {tq["query"]}\nAnswer: {r["answer"]}\nScore 1-5 JSON: {{"correctness":<int>,"grounding":<int>,"completeness":<int>}}'
        try:
            resp = generator._llm(ep)
            match = re.search(r"\{.*\}", resp, re.DOTALL)
            scores = json.loads(match.group()) if match else {"correctness":3,"grounding":3,"completeness":3}
        except: scores = {"correctness":3,"grounding":3,"completeness":3}
        results.append({"query":tq["query"], "type":tq["type"], "difficulty":tq["difficulty"],
                        "answer":r["answer"], "pipeline":label, **scores})
    df = pd.DataFrame(results)
    for m in ["correctness","grounding","completeness"]:
        print(f"  {label} {m}: {df[m].mean():.2f}/5")
    return results

print("Running retrieval evaluation...")
br_scores = eval_retrieval(TEST_QUERIES, baseline_retriever, "Baseline")
er_scores = eval_retrieval(TEST_QUERIES, enhanced_retriever, "Enhanced")
print("\nRunning generation evaluation...")
bg_scores = eval_generation(TEST_QUERIES, pipeline, False)
eg_scores = eval_generation(TEST_QUERIES, pipeline, True)

Running retrieval evaluation...


Baseline: 100%|██████████| 10/10 [00:00<00:00, 31.17it/s]


Baseline: P@K=0.287 R@K=0.300 MRR=0.300


Enhanced: 100%|██████████| 10/10 [00:12<00:00,  1.29s/it]


Enhanced: P@K=0.287 R@K=0.300 MRR=0.250

Running generation evaluation...


Gen Baseline: 100%|██████████| 10/10 [00:56<00:00,  5.63s/it]


  Baseline correctness: 4.30/5
  Baseline grounding: 3.20/5
  Baseline completeness: 3.50/5


Gen Enhanced: 100%|██████████| 10/10 [01:12<00:00,  7.22s/it]

  Enhanced correctness: 4.40/5
  Enhanced grounding: 4.80/5
  Enhanced completeness: 3.50/5


## 13. Results Comparison

In [34]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Retrieval","Generation (1-5)"])
metrics = ["Precision@K","Recall@K","MRR"]
bv = [br_scores["avg_precision"], br_scores["avg_recall"], br_scores["avg_mrr"]]
ev = [er_scores["avg_precision"], er_scores["avg_recall"], er_scores["avg_mrr"]]
fig.add_trace(go.Bar(name="Baseline",x=metrics,y=bv,marker_color="#6b7280",
    text=[f"{v:.3f}" for v in bv],textposition="outside"), row=1,col=1)
fig.add_trace(go.Bar(name="Enhanced",x=metrics,y=ev,marker_color="#4f8cff",
    text=[f"{v:.3f}" for v in ev],textposition="outside"), row=1,col=1)
gn = ["Correctness","Grounding","Completeness"]
bd, ed = pd.DataFrame(bg_scores), pd.DataFrame(eg_scores)
bg = [bd[m.lower()].mean() for m in gn]
eg = [ed[m.lower()].mean() for m in gn]
fig.add_trace(go.Bar(x=gn,y=bg,marker_color="#6b7280",text=[f"{v:.2f}" for v in bg],
    textposition="outside",showlegend=False), row=1,col=2)
fig.add_trace(go.Bar(x=gn,y=eg,marker_color="#4f8cff",text=[f"{v:.2f}" for v in eg],
    textposition="outside",showlegend=False), row=1,col=2)
fig.update_layout(template="plotly_dark",height=450,barmode="group",paper_bgcolor="#0c0e14",
    plot_bgcolor="#13151e",title="<b>Baseline vs Enhanced</b>")
fig.update_yaxes(range=[0,1.1],row=1,col=1)
fig.update_yaxes(range=[0,5.5],row=1,col=2)
fig.show()
print("\nScores by query type:")
display(ed.groupby("type")[["correctness","grounding","completeness"]].mean().round(2))


Scores by query type:


,correctness,grounding,completeness
type,,,
aggregation,4.0,5.0,3.0
analysis,4.0,5.0,4.0
change,5.0,5.0,4.0
comparison,4.0,3.0,2.0
cross_fund,4.0,5.0,3.0
edge_case,5.0,5.0,4.0
factual,4.0,5.0,4.0
sector_trend,4.0,5.0,3.0
sentiment,5.0,5.0,4.0


## 14. Demo Log

In [35]:
print("="*60+"\nDEMO LOG\n"+"="*60)
demo = []
for i, tq in enumerate(TEST_QUERIES[:5]):
    print(f"\n{'─'*60}\nDemo {i+1}/5 | {tq['type']} | {tq['difficulty']}\n{'─'*60}")
    c = pipeline.compare(tq["query"])
    commentary = f"This {tq['difficulty']}-difficulty {tq['type']} query used {c['enhanced']['num_sources']} enhanced vs {c['baseline']['num_sources']} baseline sources."
    print(f"\n📝 {commentary}")
    demo.append({"query":tq["query"],"type":tq["type"],"baseline":c["baseline"]["answer"],
                 "enhanced":c["enhanced"]["answer"],"commentary":commentary})
pd.DataFrame(demo).to_csv("demo_log.csv", index=False)
print("\n✅ Saved demo_log.csv")

DEMO LOG

────────────────────────────────────────────────────────────
Demo 1/5 | factual | easy
────────────────────────────────────────────────────────────

QUERY: What are Berkshire Hathaway's top 5 holdings?

--- BASELINE ---
Based on the provided context, the question about Berkshire Hathaway's top 5 holdings cannot be fully answered as the context only provides information about one holding, which is Bank of America Corp. However, I can provide the details about the Bank of America Corp holdings:

1. 125,720,000 shares

--- ENHANCED ---
Based on the provided sources, Berkshire Hathaway's top 5 holdings as of 2025-Q4 are [Source 1]:
1. AMERICAN EXPRESS CO (025816109): $55,145,133,598 (20.1% of the total value)
2. APPLE INC (037833100): $21,929,537,965 (8.0% of the total value)
3. COCA COLA CO (191216100): $19,765,145,984 (7.2% of th

📝 This easy-difficulty factual query used 8 enhanced vs 8 baseline sources.

────────────────────────────────────────────────────────────
Demo 2/5 | 

## 15. Reflection

### Architecture
- **Data**: SEC Bulk 13F Datasets (pre-processed TSV) — reliable, fast, all funds in one download
- **Chunking**: Hierarchical (fund overviews, positions, changes, cross-fund) vs naive baseline
- **Retrieval**: Hybrid BM25+Vector via RRF → cross-encoder reranking vs vector-only baseline
- **Generation**: Llama 3.3 70B via Groq with grounding instructions

### Failure Modes
1. **Filing lag** — 13Fs filed 45 days post-quarter; always stale
2. **Long-only** — no shorts, options, or derivatives visible
3. **Sector heuristics** — name-based classification; imperfect without CUSIP-to-GICS mapping
4. **Value ambiguity** — post-2023 values in dollars, pre-2023 in thousands
5. **Amendments** — 13F-HR/A can revise earlier filings; pipeline should deduplicate

### Improvements
- CUSIP-to-ticker mapping via OpenFIGI for proper sector/price data
- yfinance enrichment for actual position returns and Sharpe ratios
- Incremental ingestion with new-filing alerts
- Temporal embedding weighting for recency
- Options/derivatives from Schedule 13D/G filings

In [36]:
print("="*60+"\n✅ PIPELINE COMPLETE\n"+"="*60)

✅ PIPELINE COMPLETE
